## Multi Channel Processing ##

In [ ]:
import time
import numpy as np
import cupy as cp
import mne
import sys
import os

sys.path.append("../..")
import act as act_cpu
import act_gpu_cuda as act_gpu

# ---------------- SETTINGS ----------------
fs = 256
epoch_length = 256
order = 10
num_epochs = 51
num_repeats = 5

channel_sets = [
    ["FP1-F7"],
    ["FP1-F7", "F7-T7"],
    ["FP1-F7", "F7-T7", "T7-P7"],
    ["FP1-F7", "F7-T7", "T7-P7", "P7-O1"],
    ["FP1-F7", "F7-T7", "T7-P7", "P7-O1", "FP1-F3", "F3-C3", "C3-P3", "P3-O1", "FP2-F4", "F4-C4"],
]

edf_path = os.path.join(os.getcwd(), '..Testing Scripts/Data/chb01_01.edf')
fc_info       = (0.5, 45, 0.5)
logDt_info    = (-4, 1, 0.5)
c_info_fixed  = (-10, 10, 0.5)
tc_info_const = (0, epoch_length, 32)

In [4]:
print("Loading EEG...")
all_channels = list(dict.fromkeys(ch for cs in channel_sets for ch in cs))
raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
raw.pick_channels(all_channels)
raw.filter(1.0, 50.0, fir_design="firwin", fir_window="hamming",
           phase="zero", method="fir", l_trans_bandwidth=0.5, h_trans_bandwidth=12.5)
raw.notch_filter(freqs=60)
full_data = raw.get_data().T  # (samples, channels)
ch_index  = {ch: i for i, ch in enumerate(raw.ch_names)}

Loading EEG...
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 50 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 0.75 Hz)
- Upper passband edge: 50.00 Hz
- Upper transition bandwidth: 12.50 Hz (-6 dB cutoff frequency: 56.25 Hz)
- Filter length: 1691 samples (6.605 s)



/tmp/ipykernel_25736/3476691678.py:3: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)


Filtering raw data in 1 contiguous segment
Setting up band-stop filter from 59 - 61 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 59.35
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 59.10 Hz)
- Upper passband edge: 60.65 Hz
- Upper transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 60.90 Hz)
- Filter length: 1691 samples (6.605 s)



In [5]:
print("Building ACT objects (not timed)...")
act_obj_cpu = act_cpu.ACT(
    FS=fs, length=epoch_length, tc_info=tc_info_const,
    fc_info=fc_info, logDt_info=logDt_info,
    c_info=c_info_fixed, force_regenerate=True, mute=True
)
act_obj_gpu = act_gpu.ACT(
    FS=fs, length=epoch_length, tc_info=tc_info_const,
    fc_info=fc_info, logDt_info=logDt_info,
    c_info=c_info_fixed, hybrid=True, force_regenerate=True, mute=True
)

Building ACT objects (not timed)...
CPU cores: 32
GPU Devices detected: 1
Hybrid mode: True
Unified memory: False
Dictionary length: 284800
Generating dictionary on CPU...
Dictionary Generated.


In [6]:
results = {}

for channels in channel_sets:
    num_ch  = len(channels)
    col_idx = [ch_index[ch] for ch in channels]
    total_epochs = full_data.shape[0] // epoch_length
    print(f"\n{'='*50}\nChannels: {num_ch}\n{'='*50}")

    # CPU segments: list-of-channels, each a list of epochs
    segments_cpu = []
    for ci in col_idx:
        ch_data = full_data[:, ci]
        segs = [ch_data[i*epoch_length:(i+1)*epoch_length].astype(np.float32)
                for i in range(total_epochs)]
        segments_cpu.append(segs[:num_epochs])

    # GPU segments: list of batches, each shape (num_ch, epoch_length)
    segments_gpu = [
        cp.stack([cp.asarray(segments_cpu[ch][i]) for ch in range(num_ch)])
        for i in range(num_epochs)
    ]

    results[num_ch] = {"cpu": [], "gpu": []}

    for repeat in range(num_repeats):
        print(f"  repeat {repeat+1}/{num_repeats}")

        # ---- CPU: one epoch per channel sequentially ----
        epoch_times, norm_vals = [], []
        t_full_start = time.perf_counter()
        for ch_segs in segments_cpu:
            for i, seg in enumerate(ch_segs):
                t0  = time.perf_counter()
                out = act_obj_cpu.transform(seg, order=order, debug=False)
                elapsed = time.perf_counter() - t0
                norm = float(np.squeeze(out["norm_residue"]))
                if i > 0:
                    epoch_times.append(elapsed)
                    norm_vals.append(norm)
        results[num_ch]["cpu"].append({
            "full":        time.perf_counter() - t_full_start,
            "epoch_times": epoch_times,
            "norm_vals":   norm_vals,
        })

        # ---- GPU: all channels batched per epoch ----
        epoch_times, norm_vals = [], []
        t_full_start = time.perf_counter()
        for i, seg_batch in enumerate(segments_gpu):
            t0  = time.perf_counter()
            out = act_obj_gpu.transform(seg_batch, order=order, debug=False)
            cp.cuda.Stream.null.synchronize()
            elapsed = time.perf_counter() - t0
            # norm_residue shape (num_ch,) — average across channels
            norm = float(np.mean(np.squeeze(out["norm_residue"])))
            if i > 0:
                epoch_times.append(elapsed)
                norm_vals.append(norm)
        results[num_ch]["gpu"].append({
            "full":        time.perf_counter() - t_full_start,
            "epoch_times": epoch_times,
            "norm_vals":   norm_vals,
        })


Channels: 1
  repeat 1/5
  repeat 2/5
  repeat 3/5
  repeat 4/5
  repeat 5/5

Channels: 2
  repeat 1/5
  repeat 2/5
  repeat 3/5
  repeat 4/5
  repeat 5/5

Channels: 3
  repeat 1/5
  repeat 2/5
  repeat 3/5
  repeat 4/5
  repeat 5/5

Channels: 4
  repeat 1/5
  repeat 2/5
  repeat 3/5
  repeat 4/5
  repeat 5/5

Channels: 10
  repeat 1/5
  repeat 2/5
  repeat 3/5
  repeat 4/5
  repeat 5/5


In [7]:
W = 92
print(f"\n{'='*W}")
print(f"{'Device':<6} {'Ch':>4}  {'All Runtime (s)':<24} {'Epoch Runtime (s)':<26} {'Norm Residue':<24} {'Speed-up'}")
print(f"{'-'*W}")

for num_ch in sorted(results.keys()):
    cpu_reps = results[num_ch]["cpu"]
    gpu_reps = results[num_ch]["gpu"]

    def stats(reps, key):
        flat = [v for r in reps for v in (r[key] if isinstance(r[key], list) else [r[key]])]
        return np.mean(flat), np.std(flat)

    cpu_full_m,  cpu_full_s  = stats(cpu_reps, "full")
    cpu_epoch_m, cpu_epoch_s = stats(cpu_reps, "epoch_times")
    cpu_norm_m,  cpu_norm_s  = stats(cpu_reps, "norm_vals")

    gpu_full_m,  gpu_full_s  = stats(gpu_reps, "full")
    gpu_epoch_m, gpu_epoch_s = stats(gpu_reps, "epoch_times")
    gpu_norm_m,  gpu_norm_s  = stats(gpu_reps, "norm_vals")

    speedup = cpu_full_m / gpu_full_m

    print(f"{'CPU':<6} {num_ch:>4}  "
          f"{cpu_full_m:.3f} +/- {cpu_full_s:.3f}           "
          f"{cpu_epoch_m:.4f} +/- {cpu_epoch_s:.4f}         "
          f"{cpu_norm_m:.3f} +/- {cpu_norm_s:.3f}        "
          f"---")
    print(f"{'GPU':<6} {num_ch:>4}  "
          f"{gpu_full_m:.3f} +/- {gpu_full_s:.3f}           "
          f"{gpu_epoch_m:.4f} +/- {gpu_epoch_s:.4f}         "
          f"{gpu_norm_m:.3f} +/- {gpu_norm_s:.3f}        "
          f"{speedup:.2f}x")
    print(f"{'-'*W}")

print(f"{'='*W}")


Device   Ch  All Runtime (s)          Epoch Runtime (s)          Norm Residue             Speed-up
--------------------------------------------------------------------------------------------
CPU       1  3.391 +/- 0.165           0.0664 +/- 0.0096         0.325 +/- 0.133        ---
GPU       1  0.860 +/- 0.176           0.0151 +/- 0.0031         0.325 +/- 0.133        3.94x
--------------------------------------------------------------------------------------------
CPU       2  6.532 +/- 0.155           0.0640 +/- 0.0078         0.320 +/- 0.125        ---
GPU       2  1.168 +/- 0.010           0.0227 +/- 0.0041         0.320 +/- 0.111        5.59x
--------------------------------------------------------------------------------------------
CPU       3  10.064 +/- 0.191           0.0658 +/- 0.0107         0.337 +/- 0.115        ---
GPU       3  1.521 +/- 0.008           0.0296 +/- 0.0045         0.337 +/- 0.091        6.61x
--------------------------------------------------------------